In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=name,compute_cap --format=csv

Thu Oct 22 06:01:26 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%%writefile ex1_timed.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    cudaMalloc((void **)&d_a, size);
    cudaMalloc((void **)&d_b, size);
    cudaMalloc((void **)&d_c, size);

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice);

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);
    cudaDeviceSynchronize();
    dElaps = cpuSecond() - dStart;

    cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost);

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken %.7f\n", dElaps);

    free(a); free(b); free(c);
    cudaFree(d_a); cudaFree(d_b); cudaFree(d_c);
    return 0;
}

Writing ex1_timed.cu


In [3]:
!nvcc -arch=sm_75 ex1_timed.cu -o ex1_timed && ./ex1_timed

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken 0.0007322


In [4]:
%%writefile ex2_check.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define N 512

#define CHECK(call)                                                            \
do {                                                                           \
    const cudaError_t error = (call);                                          \
    if (error != cudaSuccess) {                                                \
        printf("Error: %s:%d, ", __FILE__, __LINE__);                          \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));     \
        exit(1);                                                               \
    }                                                                          \
} while (0)

__global__ void add(int *a, int *b, int *c) {
    c[blockIdx.x] = a[blockIdx.x] + b[blockIdx.x];
}

void random_ints(int *x, int size) {
    for (int i = 0; i < size; i++)
        x[i] = rand() % 100;
}

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

int main(void) {
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    int size = N * sizeof(int);
    double dStart, dElaps;

    // TODO 1: wrap each of the three cudaMalloc calls in CHECK(...)
    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int*)malloc(size); random_ints(a, N);
    b = (int*)malloc(size); random_ints(b, N);
    c = (int*)malloc(size);

    // TODO 2: wrap both host-to-device cudaMemcpy calls in CHECK(...)
    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    dStart = cpuSecond();
    add<<<N, 1>>>(d_a, d_b, d_c);

    // TODO 3: add CHECK(cudaGetLastError()); here, and wrap cudaDeviceSynchronize()
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    // TODO 4: wrap the device-to-host cudaMemcpy in CHECK(...)
    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    for (int r = 0; r < 5; r++)
        printf("%d + %d = %d\n", a[r], b[r], c[r]);

    int errors = 0;
    for (int r = 0; r < N; r++)
        if (c[r] != a[r] + b[r]) errors++;
    printf("Verification: %d mismatches out of %d\n", errors, N);
    printf("Time taken %.7f\n", dElaps);

    free(a); free(b); free(c);

    // TODO 5: wrap the three cudaFree calls in CHECK(...)
    CHECK(cudaFree(d_a));
    CHECK(cudaFree(d_b));
    CHECK(cudaFree(d_c));
    return 0;
}

Writing ex2_check.cu


In [5]:
!nvcc -arch=sm_75 ex2_check.cu -o ex2_check && ./ex2_check

83 + 77 = 160
86 + 34 = 120
77 + 90 = 167
15 + 26 = 41
93 + 24 = 117
Verification: 0 mismatches out of 512
Time taken 0.0001109


In [6]:
%%writefile ex3_picture.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define WIDTH 800   // columns
#define HEIGHT 600  // rows

#define CHECK(call)                                                            \
do {                                                                           \
    const cudaError_t error = (call);                                          \
    if (error != cudaSuccess) {                                                \
        printf("Error: %s:%d, ", __FILE__, __LINE__);                          \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));     \
        exit(1);                                                               \
    }                                                                          \
} while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void PictureKernel(unsigned char *d_Pin, unsigned char *d_Pout, int height, int width) {
    int Row = blockIdx.y * blockDim.y + threadIdx.y;
    int Col = blockIdx.x * blockDim.x + threadIdx.x;

    if ((Row < height) && (Col < width)) {
        int v = 2 * d_Pin[Row * width + Col];
        d_Pout[Row * width + Col] = (v > 255) ? 255 : v;
    }
}

int main(void) {
    size_t count = (size_t)WIDTH * HEIGHT;
    size_t size = count * sizeof(unsigned char);
    unsigned char *h_in, *h_out;
    unsigned char *d_in, *d_out;
    double dStart, dElaps;

    h_in = (unsigned char *)malloc(size);
    h_out = (unsigned char *)malloc(size);

    CHECK(cudaMalloc((void **)&d_in, size));
    CHECK(cudaMalloc((void **)&d_out, size));

    // TODO 1: give every pixel of h_in a random value between 0 and 255
    for (size_t i = 0; i < count; i++) {
        h_in[i] = (unsigned char)(rand() % 256);
    }

    CHECK(cudaMemcpy(d_in, h_in, size, cudaMemcpyHostToDevice));

    // TODO 2: 16 threads per block in BOTH x and y
    dim3 threadsPerBlock(16, 16);

    // TODO 3: blocks needed in x (columns) and in y (rows), rounding UP
    dim3 numBlocks((WIDTH + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (HEIGHT + threadsPerBlock.y - 1) / threadsPerBlock.y);

    printf("Grid: %d x %d blocks of %d x %d threads\n",
           numBlocks.x, numBlocks.y, threadsPerBlock.x, threadsPerBlock.y);

    dStart = cpuSecond();
    // TODO 4: launch PictureKernel with <<<numBlocks, threadsPerBlock>>>
    PictureKernel<<<numBlocks, threadsPerBlock>>>(d_in, d_out, HEIGHT, WIDTH);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(h_out, d_out, size, cudaMemcpyDeviceToHost));

    // TODO 5: print top-left 50x50 portion: original then updated
    printf("\n--- Top-left 50x50 Original ---\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_in[r * WIDTH + c]);
        }
        printf("\n");
    }

    printf("\n--- Top-left 50x50 Updated ---\n");
    for (int r = 0; r < 50; r++) {
        for (int c = 0; c < 50; c++) {
            printf("%3d ", h_out[r * WIDTH + c]);
        }
        printf("\n");
    }

    // TODO 6: check every pixel on the CPU (capped at 255)
    int mismatches = 0;
    int capped = 0;
    for (size_t i = 0; i < count; i++) {
        int expected = 2 * h_in[i];
        if (expected > 255) {
            expected = 255;
            capped++;
        }
        if (h_out[i] != (unsigned char)expected) {
            mismatches++;
        }
    }

    printf("\nVerification: %d mismatches out of %zu\n", mismatches, count);
    printf("Pixels capped at 255: %d\n", capped);
    printf("Time taken %.7f\n", dElaps);

    free(h_in); free(h_out);
    CHECK(cudaFree(d_in)); CHECK(cudaFree(d_out));
    return 0;
}

Writing ex3_picture.cu


In [7]:
!nvcc -arch=sm_75 ex3_picture.cu -o ex3_picture && ./ex3_picture

Grid: 50 x 38 blocks of 16 x 16 threads

--- Top-left 50x50 Original ---
103 198 105 115  81 255  74 236  41 205 186 171 242 251 227  70 124 194  84 248  27 232 231 141 118  90  46  99  51 159 201 154 102  50  13 183  49  88 163  90  37  93   5  23  88 233  94 212 171 178 
 96  34  10 167  77  49 181  91   3 160  13  34  13  71  93 205 155 135 120  86 213 112  76 156 134 234  15 152 242 235 156  83  13 167 250  90 216 176 181 219  80 194 253  93   9  90  42 165 226 163 
210 229 220  71  16 152 132 214 161  59  36  81  31  29 107 245  90 125  16 216  23 252 165  61 140  36 239 252 218 206  78 172 179  42 243 196 195 119 154 100 178 190 181 209 219  32 198  53 157 214 
158 212 187 189 162 172 238 192 141 116  54 141  24 225  34 225 154   4  34 178 109 178 216 130 145 231 176 222 132 115 155  34  71  86 223 233   2 205 169 143  65 224  28  90 193  63  59  91  67  93 
219 240 232   2 175 227 102  92  27 167 209 145 153  42 245 250 103 153 124 186 196 109  26  59 117 143  79  87 135 187  33

In [8]:
%%writefile ex4_volume.cu
#include <cstdio>
#include <cstdlib>
#include <sys/time.h>

#define NX 500  // x: columns
#define NY 500  // y: rows
#define NZ 500  // z: planes (depth)

#define CHECK(call)                                                            \
do {                                                                           \
    const cudaError_t error = (call);                                          \
    if (error != cudaSuccess) {                                                \
        printf("Error: %s:%d, ", __FILE__, __LINE__);                          \
        printf("code:%d, reason: %s\n", error, cudaGetErrorString(error));     \
        exit(1);                                                               \
    }                                                                          \
} while (0)

double cpuSecond() {
    struct timeval tp;
    gettimeofday(&tp, NULL);
    return (double)tp.tv_sec + (double)tp.tv_usec * 1.e-6;
}

__global__ void mulElem3D(int *a, int *b, int *c, int nx, int ny, int nz) {
    // TODO 1: the x, y and z coordinates of this thread
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    int z = blockIdx.z * blockDim.z + threadIdx.z;

    // TODO 2: only compute if x, y AND z are all inside the volume
    if (x < nx && y < ny && z < nz) {
        // TODO 3: flat index of (x, y, z): ((z * ny + y) * nx + x)
        size_t idx = ((size_t)z * ny + y) * nx + x;
        c[idx] = a[idx] * b[idx];
    }
}

void random_ints(int *x, size_t size) {
    for (size_t i = 0; i < size; i++)
        x[i] = rand() % 100;
}

int main(void) {
    size_t count = (size_t)NX * NY * NZ;  // 125,000,000 elements
    size_t size = count * sizeof(int);     // 500 MB per array
    int *a, *b, *c;
    int *d_a, *d_b, *d_c;
    double dStart, dElaps;

    CHECK(cudaMalloc((void **)&d_a, size));
    CHECK(cudaMalloc((void **)&d_b, size));
    CHECK(cudaMalloc((void **)&d_c, size));

    a = (int*)malloc(size); random_ints(a, count);
    b = (int*)malloc(size); random_ints(b, count);
    c = (int*)malloc(size);

    CHECK(cudaMemcpy(d_a, a, size, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(d_b, b, size, cudaMemcpyHostToDevice));

    // 8 x 8 x 8 = 512 threads per block
    dim3 threadsPerBlock(8, 8, 8);

    // TODO 4: blocks needed in x, y and z, rounding UP
    dim3 numBlocks((NX + threadsPerBlock.x - 1) / threadsPerBlock.x,
                   (NY + threadsPerBlock.y - 1) / threadsPerBlock.y,
                   (NZ + threadsPerBlock.z - 1) / threadsPerBlock.z);

    printf("Grid: %d x %d x %d blocks\n", numBlocks.x, numBlocks.y, numBlocks.z);

    dStart = cpuSecond();
    // TODO 5: launch mulElem3D with <<<numBlocks, threadsPerBlock>>>
    mulElem3D<<<numBlocks, threadsPerBlock>>>(d_a, d_b, d_c, NX, NY, NZ);
    CHECK(cudaGetLastError());
    CHECK(cudaDeviceSynchronize());
    dElaps = cpuSecond() - dStart;

    CHECK(cudaMemcpy(c, d_c, size, cudaMemcpyDeviceToHost));

    // TODO 6: print the last 1000 elements of C in memory order
    // Starts at z = 499, y = 498, x = 0 up to z = 499, y = 499, x = 499
    printf("\nPrinting sample of last 1000 elements:\n");
    size_t start_idx = count - 1000;
    size_t z_start = start_idx / ((size_t)NY * NX);
    size_t rem = start_idx % ((size_t)NY * NX);
    size_t y_start = rem / NX;
    size_t x_start = rem % NX;

    printf("First of last 1000: C[%zu][%zu][%zu] = %d x %d = %d\n",
           z_start, y_start, x_start, a[start_idx], b[start_idx], c[start_idx]);
    printf("Last result: C[499][499][499] = %d x %d = %d\n",
           a[count - 1], b[count - 1], c[count - 1]);

    // TODO 7: check all elements on the CPU
    int mismatches = 0;
    for (size_t i = 0; i < count; i++) {
        if (c[i] != a[i] * b[i]) {
            mismatches++;
        }
    }

    printf("Verification: %d mismatches out of %zu\n", mismatches, count);
    printf("Time taken %.7f\n", dElaps);

    free(a); free(b); free(c);
    CHECK(cudaFree(d_a)); CHECK(cudaFree(d_b)); CHECK(cudaFree(d_c));
    return 0;
}

Writing ex4_volume.cu


In [9]:
!nvcc -arch=sm_75 ex4_volume.cu -o ex4_volume && ./ex4_volume

Grid: 63 x 63 x 63 blocks

Printing sample of last 1000 elements:
First of last 1000: C[499][498][0] = 76 x 15 = 1140
Last result: C[499][499][499] = 47 x 68 = 3196
Verification: 0 mismatches out of 125000000
Time taken 0.0078988


In [10]:
!zip -r SE3082_Lab10_CUDA_Files.zip *.cu

  adding: ex1_timed.cu (deflated 58%)
  adding: ex2_check.cu (deflated 65%)
  adding: ex3_picture.cu (deflated 65%)
  adding: ex4_volume.cu (deflated 64%)
